In [14]:
from fitparse import FitFile
from pathlib import Path
from collections import defaultdict

def discover_all_fit_fields(folder_path):
    """Discover ALL possible fields across all FIT files in a folder."""

    folder = Path(folder_path)
    fit_files = list(folder.glob('*.[fF][iI][tT]'))

    print(f"Analyzing {len(fit_files)} FIT files...\n")

    # Store all message types and their fields
    all_messages = defaultdict(set)

    for fit_file in fit_files:
        try:
            fitfile = FitFile(str(fit_file))

            for message in fitfile.get_messages():
                message_name = message.name

                # Collect all field names for this message type
                for data in message:
                    if data.value is not None:  # Only include fields with values
                        all_messages[message_name].add(data.name)

        except Exception as e:
            print(f"Error reading {fit_file.name}: {e}")

    # Display results
    print("=" * 80)
    print("ALL AVAILABLE MESSAGE TYPES AND THEIR FIELDS")
    print("=" * 80)

    for message_type in sorted(all_messages.keys()):
        fields = sorted(all_messages[message_type])
        print(f"\n📦 {message_type.upper()} ({len(fields)} fields)")
        print("-" * 80)
        for field in fields:
            print(f"   • {field}")

    return all_messages


def show_sample_values(folder_path, message_type=None, limit=3):
    """Show sample values for fields (optionally filter by message type)."""

    folder = Path(folder_path)
    fit_files = list(folder.glob('*.[fF][iI][tT]'))[:limit]  # Only check first few files

    print(f"\n{'=' * 80}")
    print(f"SAMPLE VALUES (from first {limit} files)")
    print(f"{'=' * 80}\n")

    for fit_file in fit_files:
        print(f"\n📄 {fit_file.name}")
        print("-" * 80)

        try:
            fitfile = FitFile(str(fit_file))

            for message in fitfile.get_messages():
                # Skip if filtering by message type
                if message_type and message.name != message_type:
                    continue

                values = []
                for data in message:
                    if data.value is not None:
                        values.append(f"{data.name}={data.value}")

                if values:
                    print(f"\n  {message.name}:")
                    for v in values[:10]:  # Show max 10 fields per message
                        print(f"    {v}")

        except Exception as e:
            print(f"  Error: {e}")


# RUN IT
folder_path = 'SpecialBike'  # Your folder

# 1. Discover all available fields
all_fields = discover_all_fit_fields(folder_path)
print(all_fields)

# 2. Show sample values (optional - comment out if too much output)
#show_sample_values(folder_path, limit=2)

# 3. If you want to see specific message types in detail:
# show_sample_values(folder_path, message_type='record', limit=1)
# show_sample_values(folder_path, message_type='session', limit=1)

Analyzing 26 FIT files...

ALL AVAILABLE MESSAGE TYPES AND THEIR FIELDS

📦 ACTIVITY (4 fields)
--------------------------------------------------------------------------------
   • local_timestamp
   • num_sessions
   • timestamp
   • total_timer_time

📦 DEVELOPER_DATA_ID (2 fields)
--------------------------------------------------------------------------------
   • application_id
   • developer_data_index

📦 FILE_ID (4 fields)
--------------------------------------------------------------------------------
   • manufacturer
   • product_name
   • time_created
   • type

📦 HRV (1 fields)
--------------------------------------------------------------------------------
   • time

📦 LAP (29 fields)
--------------------------------------------------------------------------------
   • avg_cadence
   • avg_grade
   • avg_heart_rate
   • avg_power
   • avg_speed
   • enhanced_avg_speed
   • enhanced_max_altitude
   • enhanced_max_speed
   • enhanced_min_altitude
   • intensity
   • left_righ

In [15]:
from fitparse import FitFile
from pathlib import Path
from collections import defaultdict
import pandas as pd

def get_fit_summary(folder_path):
    """Extract summary from all FIT files in a folder."""

    # Find all FIT files (only in main folder, not subfolders)
    folder = Path(folder_path)
    fit_files = list(folder.glob('*.[fF][iI][tT]'))

    print(f"Found {len(fit_files)} FIT files in folder")

    all_data = []

    for fit_file in fit_files:
        file_info = {
            'file_name': fit_file.name,
            'friendly_name': None,
            'sport': None,
            'sub_sport': None,
            'gender': None,
            'resting_heart_rate': None,
            'weight': None,
            'age': None,
            'height': None
        }

        try:
            fitfile = FitFile(str(fit_file))

            for message in fitfile.get_messages():
                if message.name == 'session':
                    for data in message:
                        if data.name == 'sport' and data.value is not None:
                            file_info['sport'] = data.value
                        elif data.name == 'sub_sport' and data.value is not None:
                            file_info['sub_sport'] = data.value

                elif message.name == 'sport':
                    for data in message:
                        if data.name == 'sport' and data.value is not None:
                            file_info['sport'] = data.value
                        elif data.name == 'sub_sport' and data.value is not None:
                            file_info['sub_sport'] = data.value

                elif message.name == 'user_profile':
                    for data in message:
                        if data.value is not None:
                            if data.name == 'friendly_name':
                                file_info['friendly_name'] = data.value
                            elif data.name == 'gender':
                                file_info['gender'] = data.value
                            elif data.name == 'resting_heart_rate':
                                file_info['resting_heart_rate'] = data.value
                            elif data.name == 'weight':
                                file_info['weight'] = data.value

        except Exception as e:
            print(f"Error: {fit_file.name}: {e}")

        all_data.append(file_info)

    df = pd.DataFrame(all_data)

    # Summary
    total_files = len(df)
    sport_counts = df['sport'].value_counts().to_dict()

    # Group by person AND sport/sub_sport combination (separate row for each combo)
    person_groups = df.groupby(['friendly_name', 'sport', 'sub_sport']).agg({
        'file_name': ['count', lambda x: list(x)],  # Count and list of filenames
        'gender': 'first',
        'resting_heart_rate': 'first',
        'weight': 'first'
    })

    # Flatten column names
    person_groups.columns = ['file_count', 'session_files', 'gender', 'resting_heart_rate', 'weight']
    person_groups = person_groups.reset_index()

    # Reorder columns: name, sport, sub_sport, then the rest
    column_order = ['friendly_name', 'sport', 'sub_sport', 'file_count', 'session_files',
                    'gender', 'resting_heart_rate', 'weight']
    person_groups = person_groups[column_order]

    print(f"\n📊 TOTAL FILES: {total_files}")
    print(f"\n⚽ FILES PER SPORT:")
    for sport, count in sorted(sport_counts.items(), key=lambda x: x[1], reverse=True):
        print(f"   {sport}: {count} files")

    print(f"\n👥 FILES PER PERSON (by sport/sub_sport):\n")
    display(person_groups)

    return {'total_files': total_files, 'sport_counts': sport_counts,
            'person_summary': person_groups, 'detailed_data': df}

# RUN IT
folder_path = 'SpecialBike'  # CHANGE THIS
summary = get_fit_summary(folder_path)

Found 26 FIT files in folder

📊 TOTAL FILES: 26

⚽ FILES PER SPORT:
   cycling: 24 files
   rowing: 2 files

👥 FILES PER PERSON (by sport/sub_sport):



,friendly_name,sport,sub_sport,file_count,session_files,gender,resting_heart_rate,weight
0,Julia Walkenhorst,cycling,indoor_cycling,4,[14691322045_Zwift__3x12min_3030_4x2min_Vo2Max...,female,50.0,64.0
1,Lukas Feth,cycling,indoor_cycling,1,[i90773880_Zwift__9x_10_3030.fit],male,41.0,59.5
2,Martin Maertens,cycling,indoor_cycling,4,[12376609525_Zwift__HIT__HIT_anaerob_6x30sec_i...,male,48.0,63.0
3,Niklas von Haaren,cycling,indoor_cycling,2,[10410929216_Zwift__10x_3030__8x_4040__4x_6060...,male,56.0,84.1
4,Oli W,cycling,indoor_cycling,4,[10577414184_Zwift__HIT__HIT_LI_4x8_min_1_in_W...,male,NaN,75.0
5,Rocko,cycling,indoor_cycling,5,[i93604453_Zwift__HIT__IE_3x13x3015_in_Watopia...,male,45.0,71.0
6,Steffen_pacman,cycling,indoor_cycling,1,[10283878603_Zwift__HIT__HIT_EB_5x3_min_in_Wat...,male,40.0,71.6
7,Timon0509,cycling,generic,1,[i79840180_AP08_Anaerobic_Power_60_Second_Repe...,male,NaN,70.0
8,awenjb,rowing,indoor_rowing,2,"[14612015090_3x_6x_3030.fit, 15097759724_3x_8_...",male,NaN,70.0
9,saaraah3107,cycling,generic,2,"[i79851276_Glattbach__3030.fit, i79851294_Groo...",female,42.0,61.5


In [18]:
# Save to Excel
output_file = 'specialbike_fit_summary.xlsx'
summary['person_summary'].to_excel(output_file, index=False)
print(f"\n✅ Saved to {output_file}")



✅ Saved to specialbike_fit_summary.xlsx
